# HW3 — Human / AI / Fusion text classification

All logic is in `hw3_text_detection.py`; this notebook runs the pipeline stage by stage.
Runs on **Google Colab** and **locally** (VS Code) without changes: paths and device are detected automatically.

Heavy stages save their results to `runs/<name>/` on Google Drive. Set a toggle to `False`
to skip a stage and load its saved results instead.

## 0. Setup

In [ ]:
# ---------------- toggles ----------------
RUN_EDA = True
RECOMPUTE_GROUPS = False   # cached in splits/groups.csv
RECOMPUTE_SPLIT = False    # cached in splits/split_v1.csv (always recomputed with new groups)
RUN_BASELINE = True        # False -> only load runs/baseline_tfidf (models already saved are never refitted)
CACHE_FEATURES = True      # save TF-IDF matrices to runs/baseline_tfidf/features (~3-4 GB on Drive)
RUN_ENCODER = True         # False -> load runs/encoder_xlmr_chunks

# experiments (sections 8-13); False -> only show results already saved in runs/
RUN_EXP_LONG_TEXT = True     # 1. XLM-R: how window predictions are combined (GPU, inference only)
RUN_EXP_WINDOW_META = True   # 2. disagreement between windows -> small model (uses exp 1 predictions)
RUN_EXP_STYLOMETRY = True    # 3. stylometric features -> gradient boosting (CPU)
RUN_EXP_PERPLEXITY = True    # 4. perplexity features from Qwen2.5-0.5B -> gradient boosting (GPU)
RUN_EXP_MDEBERTA = True      # 5. mDeBERTa-v3-base, same training as XLM-R (GPU, long)
RUN_EXP_STACKING = True      # 6. class thresholds + stacking of all saved runs (CPU)

SUBSAMPLE_N = 40_000       # fixed subsample of whole groups; 20% of it is validation
BASELINE_RUN = "baseline_tfidf"
ENCODER_RUN = "encoder_xlmr_chunks"

# ---------------- environment ----------------
REPO_URL = "https://github.com/Rein-h/pavlunyshyn_hw3_AI.git"
LOCAL_ROOT = r"G:\My Drive\hw3"                        # local run only: folder that contains data/

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if os.path.exists(REPO_DIR):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
    %pip install -q -r {REPO_DIR}/requirements.txt
    sys.path.insert(0, REPO_DIR)

In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import hw3_text_detection as hw
importlib.reload(hw)  # picks up a new version of the .py after git pull

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
hw.seed_everything(hw.SEED)
RECOMPUTE_SPLIT = RECOMPUTE_SPLIT or RECOMPUTE_GROUPS

ENC_CFG = hw.EncoderConfig(
    model_name="xlm-roberta-base",
    max_len=512,          # tokens per window
    overlap=64,           # overlap of neighbouring windows
    train_windows=1,      # random windows per text per epoch
    max_eval_windows=8,   # windows averaged per text at inference
    epochs=2,
    lr=2e-5,
    freeze_embeddings=True,   # embedding layer is not trained: checkpoint ~1 GB instead of ~3.3 GB
    checkpoint_minutes=15,    # save progress every 15 min inside an epoch
)

paths = hw.get_paths(local_root=LOCAL_ROOT, colab_root="/content/drive/MyDrive/hw3")
hw.describe_environment()

## 1. Data

In [ ]:
train, test, sample = hw.load_data(paths)
hw.data_overview(train, test, sample)

## 2. EDA
### 2.1 Class balance by language and domain

In [ ]:
if RUN_EDA:
    tables = hw.class_balance_tables(train)
    display(tables["label"], tables["lang x label"], tables["domain x label"])
    hw.plot_class_balance(train)
    plt.show()

### 2.2 Text lengths

In [ ]:
if RUN_EDA:
    hw.plot_lengths(train)
    plt.show()
    display(hw.length_table(train))

### 2.3 Length in encoder tokens
Share of texts that do not fit into one encoder window — the reason the encoder works on windows (chunks).

In [ ]:
if RUN_EDA:
    display(hw.token_length_stats(train["text"], ENC_CFG.model_name, ENC_CFG.max_len))

### 2.4 Examples of each class

In [ ]:
if RUN_EDA:
    hw.show_examples(train, n_chars=300)

### 2.5 Train vs. test distribution

In [ ]:
if RUN_EDA:
    display(hw.compare_train_test(train, test))

## 3. Groups of versions and the fixed validation split
A human text and its ai / fusion versions are near-duplicates and are always on the same side of the
train/test split. Groups are rebuilt from shared word 5-grams; the validation split keeps every group
whole, so validation mimics the test split.

In [ ]:
train["group"] = hw.load_or_make_groups(train, paths, recompute=RECOMPUTE_GROUPS)
hw.group_composition(train)

In [ ]:
tr, va = hw.load_or_make_split(train, paths, n_samples=SUBSAMPLE_N, recompute=RECOMPUTE_SPLIT)
# the same fingerprint in every session = exactly the same train/val rows
print("split fingerprint:", hw.split_fingerprint(tr, va, test))
hw.split_summary(tr, va)

### 3.1 What makes fusion different
Every ai / fusion text that has a human version in its group is compared with that human version:
overlap of word 3-grams, share of sentences copied verbatim, relative length.

In [ ]:
if RUN_EDA:
    cmp = hw.compare_with_human_version(train)
    display(cmp.groupby("label").median().round(3))
    hw.plot_human_version_comparison(cmp)
    plt.show()
    hw.show_group_example(train, n_chars=400)

### Free memory
From here on only the subsample (`tr`, `va`) and `test` are needed. The full train set (~272k texts)
is removed from RAM; to use it again, rerun the notebook from section 1.

In [ ]:
import gc
del train
gc.collect()

## 4. Baseline: TF-IDF + linear models
TF-IDF is fitted on the train split only. Small grid search (feature set × model × C) on the fixed validation split.

Every model is saved to `runs/baseline_tfidf/grid/<features>_<model>_C<C>/` (metrics + val/test probabilities)
right after it is fitted; TF-IDF matrices are cached in `runs/baseline_tfidf/features/`.
The grid runs in three cells, one per feature set. After a crash, rerun from the cell that crashed: saved models are skipped.

In [ ]:
BL_DIR = paths.runs / BASELINE_RUN
BASELINE_MODELS = {"logreg": [1.0, 10.0], "linsvc": [0.1, 1.0]}   # model -> values of C

def run_feature_set(features):
    if RUN_BASELINE:
        hw.run_baseline_grid(tr, va, test, BL_DIR, cache_features=CACHE_FEATURES,
                             grid={"features": [features], "models": BASELINE_MODELS})

### 4.1 Word n-grams (1–2)

In [ ]:
run_feature_set("word")

### 4.2 Character n-grams (3–5)

In [ ]:
run_feature_set("char")

### 4.3 Word + character n-grams

In [ ]:
run_feature_set("word+char")

### 4.4 Grid results and the best baseline
The best model of the grid is copied to `runs/baseline_tfidf/` (metrics, probabilities, `submission.csv`).

In [ ]:
bl_config = hw.finalize_baseline(BL_DIR, va, test, sample)
bl_run = hw.load_run(BL_DIR)
pd.read_csv(BL_DIR / "grid.csv")

In [ ]:
bl_per_class, bl_by, _ = hw.run_report(bl_run, va, f"Baseline {bl_config['model_id']}")
plt.show()
display(bl_per_class, bl_by)

## 5. Encoder: fine-tuned multilingual transformer
Long texts are split into overlapping windows of `max_len` tokens. Training sees a random window of
each text (a new one every epoch); prediction averages the probabilities of up to `max_eval_windows`
windows that cover the text. The best epoch is selected on the same validation split as the baseline.

If the Colab session dies, run the notebook again: training resumes from the last checkpoint (saved every `checkpoint_minutes`).

In [ ]:
ENC_DIR = paths.runs / ENCODER_RUN
if RUN_ENCODER:
    enc = hw.train_encoder(tr, va, test, ENC_CFG, ENC_DIR)
    enc_sub = hw.make_submission(test["id"], np.array(hw.LABELS)[enc["test_proba"].argmax(1)], sample)
    hw.save_run(ENC_DIR, enc["metrics"], enc["config"], va["id"], enc["val_proba"],
                test["id"], enc["test_proba"], enc_sub)

if hw.run_exists(paths, ENCODER_RUN):
    enc_run = hw.load_run(ENC_DIR)
    display(pd.DataFrame(enc_run["metrics"]["history"]))
else:
    print("No saved encoder run yet - set RUN_ENCODER = True.")

In [ ]:
if hw.run_exists(paths, ENCODER_RUN):
    enc_per_class, enc_by, _ = hw.run_report(enc_run, va, f"Encoder {ENC_CFG.model_name}")
    plt.show()
    display(enc_per_class, enc_by)

## 6. Baseline vs. encoder (same validation split)
Submission files: `runs/<run name>/submission.csv` on Google Drive.

In [ ]:
hw.compare_runs(paths, [BASELINE_RUN, ENCODER_RUN], va)

# 7. Experiments
Every experiment is saved to `runs/<name>/` (metrics, val/test probabilities, `submission.csv`) and compared
with the TF-IDF baseline and the XLM-R encoder (the fixed reference) on the same validation split.

Models that learn from other models' validation outputs (experiments 2 and 6) are trained on the validation
set with **5-fold cross-validation by groups**: their val score is always measured on rows they were not trained on.

In [ ]:
import dataclasses

FOLDS = hw.grouped_folds(va)                      # 5 folds of val, groups kept whole
LANGS, DOMAINS = sorted(tr["lang"].unique()), sorted(tr["domain"].unique())
EXP_COMPARE = [BASELINE_RUN, ENCODER_RUN]          # every stage is compared with these two

## 8. Experiment 1 — long-text handling
The trained XLM-R predicts **every** window of every text once; the window predictions are cached
(`runs/encoder_xlmr_chunks/windows_<split>.npz`). Three ways to turn them into one prediction per text:
only the first window (= plain truncation), first + last window, mean of all windows.
No retraining.

In [ ]:
LONG_RUNS = {f"xlmr_long_{how}": how for how in ("first", "first_last", "mean")}
if RUN_EXP_LONG_TEXT or RUN_EXP_WINDOW_META:
    win_val = hw.encoder_window_predictions(ENC_DIR, va, "val")
    n_win = np.bincount(win_val["doc"], minlength=len(va))
    print(f"val texts with more than one window: {(n_win > 1).mean():.1%}, max windows: {n_win.max()}")

In [ ]:
if RUN_EXP_LONG_TEXT or RUN_EXP_WINDOW_META:
    win_test = hw.encoder_window_predictions(ENC_DIR, test, "test")

In [ ]:
if RUN_EXP_LONG_TEXT:
    for name, how in LONG_RUNS.items():
        hw.save_experiment(paths, name, va, test,
                           hw.aggregate_windows(win_val, len(va), how),
                           hw.aggregate_windows(win_test, len(test), how), sample,
                           {"encoder_run": ENCODER_RUN, "aggregation": how})
hw.compare_runs(paths, EXP_COMPARE + list(LONG_RUNS), va)

## 9. Experiment 2 — disagreement between windows
Fusion = a human text partly rewritten by an LLM, so its windows may disagree: some look human, some look ai.
Features per text from its window predictions: mean / max / min / std / range of every class probability,
first and last window, number of windows, + language and domain. A gradient-boosting model replaces the plain mean.

In [ ]:
WIN_RUN = "xlmr_window_meta"
if RUN_EXP_WINDOW_META:
    Xw_val = hw.add_meta_columns(hw.window_disagreement_features(win_val, len(va)), va, LANGS, DOMAINS)
    Xw_test = hw.add_meta_columns(hw.window_disagreement_features(win_test, len(test)), test, LANGS, DOMAINS)
    win_oof, win_test_p, win_model = hw.cv_on_val(Xw_val, va["label"], FOLDS, Xw_test, hw.make_hgb)
    hw.save_experiment(paths, WIN_RUN, va, test, win_oof, win_test_p, sample,
                       {"encoder_run": ENCODER_RUN, "model": "HistGradientBoosting",
                        "val_proba": "out-of-fold, 5-fold group CV on val"})
hw.compare_runs(paths, EXP_COMPARE + ["xlmr_long_mean", WIN_RUN], va)

In [ ]:
if RUN_EXP_WINDOW_META:
    _, win_imp = hw.plot_permutation_importance(
        win_model, Xw_val, va["label"], "Window-disagreement model (fitted on val, in-sample)")
    plt.show()

## 10. Experiment 3 — stylometry
How a text is written, not what it is about: sentence length and its variation, word length, vocabulary
richness, punctuation, digits, capital letters, first-person pronouns, and how much the style differs between
the two halves of the text. Gradient boosting trained on the train split.

In [ ]:
STYLO_RUN = "stylometry_hgb"
STYLO_DIR = paths.runs / STYLO_RUN
if RUN_EXP_STYLOMETRY:
    Xs = {name: hw.load_or_make_stylometry(df, STYLO_DIR / f"features_{name}.pkl")
          for name, df in (("train", tr), ("val", va), ("test", test))}
    print(f"{Xs['train'].shape[1]} stylometric features")

In [ ]:
if RUN_EXP_STYLOMETRY:
    show = ["sent_len_mean", "sent_len_cv", "word_len_mean", "ttr300", "comma_per1k", "emdash_per1k",
            "first_person_per100", "halves_diff_sent_len_mean", "chunk_sent_len_std"]
    display(Xs["train"][show].groupby(tr["label"].to_numpy()).median().T.round(3))

In [ ]:
if RUN_EXP_STYLOMETRY:
    Xs_tr = hw.add_meta_columns(Xs["train"], tr, LANGS, DOMAINS)
    Xs_va = hw.add_meta_columns(Xs["val"], va, LANGS, DOMAINS)
    Xs_te = hw.add_meta_columns(Xs["test"], test, LANGS, DOMAINS)
    stylo_model = hw.make_hgb().fit(Xs_tr, tr["label"])
    hw.save_experiment(paths, STYLO_RUN, va, test, hw.model_proba(stylo_model, Xs_va),
                       hw.model_proba(stylo_model, Xs_te), sample,
                       {"model": "HistGradientBoosting", "features": list(Xs_tr.columns)})
hw.compare_runs(paths, EXP_COMPARE + [STYLO_RUN], va)

In [ ]:
if RUN_EXP_STYLOMETRY:
    _, stylo_imp = hw.plot_permutation_importance(stylo_model, Xs_va, va["label"],
                                                  "Stylometry model: feature importance on val")
    plt.show()

## 11. Experiment 4 — perplexity features
An open causal language model (Qwen2.5-0.5B, English + German) reads every text (first 1024 tokens) and gives,
for each token, how probable the actual token was and how uncertain the model was (entropy).
Features: mean / spread / percentiles of the token log-probabilities, entropy, their ratio, and how
predictability changes between 64-token segments and between the two halves of the text (the fusion signal).
The LM is not trained; a gradient-boosting model is trained on the features of a fixed train subset.

Features are computed in shards of 2000 texts saved to `runs/perplexity_qwen/features/`; after a crash,
rerun the cell and it continues with the next shard.

In [ ]:
PPL_RUN = "perplexity_qwen"
PPL_DIR = paths.runs / PPL_RUN
PPL_CFG = hw.PerplexityConfig(model_name="Qwen/Qwen2.5-0.5B", max_tokens=1024, batch_size=4,
                              train_n=16_000)
ppl_tr = (tr.sample(PPL_CFG.train_n, random_state=hw.SEED).reset_index(drop=True)
          if PPL_CFG.train_n and PPL_CFG.train_n < len(tr) else tr)

In [ ]:
if RUN_EXP_PERPLEXITY:
    Xp_va = hw.load_or_make_perplexity(va, PPL_CFG, PPL_DIR / "features", "val")

In [ ]:
if RUN_EXP_PERPLEXITY:
    Xp_te = hw.load_or_make_perplexity(test, PPL_CFG, PPL_DIR / "features", "test")

In [ ]:
if RUN_EXP_PERPLEXITY:
    Xp_tr = hw.load_or_make_perplexity(ppl_tr, PPL_CFG, PPL_DIR / "features", "train")

In [ ]:
if RUN_EXP_PERPLEXITY:
    show = ["lp_mean", "lp_std", "ent_mean", "lp_over_ent", "seg_lp_std", "seg_lp_range", "halves_lp_diff"]
    display(Xp_va[show].groupby(va["label"].to_numpy()).median().T.round(3))

In [ ]:
if RUN_EXP_PERPLEXITY:
    Xp_tr_m = hw.add_meta_columns(Xp_tr, ppl_tr, LANGS, DOMAINS)
    Xp_va_m = hw.add_meta_columns(Xp_va, va, LANGS, DOMAINS)
    Xp_te_m = hw.add_meta_columns(Xp_te, test, LANGS, DOMAINS)
    ppl_model = hw.make_hgb().fit(Xp_tr_m, ppl_tr["label"])
    hw.save_experiment(paths, PPL_RUN, va, test, hw.model_proba(ppl_model, Xp_va_m),
                       hw.model_proba(ppl_model, Xp_te_m), sample,
                       {"lm": dataclasses.asdict(PPL_CFG), "model": "HistGradientBoosting"})
hw.compare_runs(paths, EXP_COMPARE + [PPL_RUN], va)

In [ ]:
if RUN_EXP_PERPLEXITY:
    _, ppl_imp = hw.plot_permutation_importance(ppl_model, Xp_va_m, va["label"],
                                                "Perplexity model: feature importance on val")
    plt.show()

## 12. Experiment 5 — another backbone: mDeBERTa-v3-base
Exactly the same training as XLM-R (same split, windows, epochs, learning rate, checkpoints); only the
pretrained model changes. If the session dies, run the notebook again: training resumes from the last checkpoint.

In [ ]:
MDEB_RUN = "encoder_mdeberta_chunks"
MDEB_DIR = paths.runs / MDEB_RUN
MDEB_CFG = dataclasses.replace(ENC_CFG, model_name="microsoft/mdeberta-v3-base",
                               batch_size=None, eval_batch_size=None)
if RUN_EXP_MDEBERTA:
    mdeb = hw.train_encoder(tr, va, test, MDEB_CFG, MDEB_DIR)
    mdeb_sub = hw.make_submission(test["id"], np.array(hw.LABELS)[mdeb["test_proba"].argmax(1)], sample)
    hw.save_run(MDEB_DIR, mdeb["metrics"], mdeb["config"], va["id"], mdeb["val_proba"],
                test["id"], mdeb["test_proba"], mdeb_sub)
if hw.run_exists(paths, MDEB_RUN):
    display(pd.DataFrame(hw.load_run(MDEB_DIR)["metrics"]["history"]))

In [ ]:
hw.compare_runs(paths, EXP_COMPARE + [MDEB_RUN], va)

## 13. Experiment 6 — class thresholds and stacking
* **Thresholds:** a bias is added to the log-probabilities of `ai` and `fusion` to maximise macro-F1.
  Each val fold uses a bias tuned on the other folds (honest estimate); test uses the bias tuned on all of val.
* **Stacking:** a logistic regression learns how to combine the log-probabilities of all saved runs
  (+ language and domain), trained with 5-fold group CV on val.
* **Stacking + thresholds:** both together — the candidate for the final submission.

In [ ]:
THR_RUN, STACK_RUN, STACK_THR_RUN = "xlmr_thresholds", "stack", "stack_thresholds"
if RUN_EXP_STACKING:
    ref_val, ref_test = hw.run_probs(paths, ENCODER_RUN, va, test)
    thr_val, thr_test, thr_bias = hw.cv_class_bias(ref_val, va["label"], FOLDS, ref_test)
    print("bias (human, ai, fusion) tuned on all of val:", thr_bias.round(2))
    hw.save_experiment(paths, THR_RUN, va, test, thr_val, thr_test, sample,
                       {"base_run": ENCODER_RUN, "bias": thr_bias.tolist(),
                        "val_proba": "bias tuned on the other folds (5-fold group CV on val)"})

In [ ]:
STACK_INPUTS = [BASELINE_RUN, ENCODER_RUN, WIN_RUN, MDEB_RUN, STYLO_RUN, PPL_RUN]
if RUN_EXP_STACKING:
    Xst_va, Xst_te, stack_used = hw.stacking_inputs(paths, STACK_INPUTS, va, test)
    print("stacking inputs:", stack_used)
    Xst_va = hw.add_meta_columns(Xst_va, va, LANGS, DOMAINS)
    Xst_te = hw.add_meta_columns(Xst_te, test, LANGS, DOMAINS)
    stack_oof, stack_test, stacker = hw.cv_on_val(Xst_va, va["label"], FOLDS, Xst_te, hw.make_stacker)
    hw.save_experiment(paths, STACK_RUN, va, test, stack_oof, stack_test, sample,
                       {"inputs": stack_used, "model": "StandardScaler + LogisticRegression(C=1)",
                        "val_proba": "out-of-fold, 5-fold group CV on val"})

In [ ]:
if RUN_EXP_STACKING:
    sthr_val, sthr_test, sthr_bias = hw.cv_class_bias(stack_oof, va["label"], FOLDS, stack_test)
    print("bias (human, ai, fusion) tuned on all of val:", sthr_bias.round(2))
    hw.save_experiment(paths, STACK_THR_RUN, va, test, sthr_val, sthr_test, sample,
                       {"base_run": STACK_RUN, "bias": sthr_bias.tolist(),
                        "val_proba": "bias tuned on the other folds (5-fold group CV on val)"})
hw.compare_runs(paths, EXP_COMPARE + [THR_RUN, STACK_RUN, STACK_THR_RUN], va)

# 14. Final comparison (same validation split)
`Δ vs baseline` / `Δ vs XLM-R` = difference in val macro-F1. The table is also saved to `runs/final_comparison.csv`;
the submission of every run is `runs/<name>/submission.csv`.

In [ ]:
ALL_RUNS = {
    BASELINE_RUN: "TF-IDF word+char + LogReg (train -> val)",
    ENCODER_RUN: "XLM-R, mean of up to 8 windows (reference)",
    "xlmr_long_first": "Exp 1: XLM-R, first window only",
    "xlmr_long_first_last": "Exp 1: XLM-R, first + last window",
    "xlmr_long_mean": "Exp 1: XLM-R, mean of all windows",
    WIN_RUN: "Exp 2: window disagreement -> HGB (CV on val)",
    STYLO_RUN: "Exp 3: stylometry -> HGB (train -> val)",
    PPL_RUN: "Exp 4: Qwen2.5-0.5B perplexity -> HGB (train subset -> val)",
    MDEB_RUN: "Exp 5: mDeBERTa-v3-base (train -> val)",
    THR_RUN: "Exp 6: XLM-R + class thresholds (CV on val)",
    STACK_RUN: "Exp 6: stacking of all runs (CV on val)",
    STACK_THR_RUN: "Exp 6: stacking + class thresholds (CV on val)",
}
final = hw.compare_runs(paths, ALL_RUNS, va)
final.to_csv(paths.runs / "final_comparison.csv")
final